Objective:\
Create clean, trusted datasets for modeling.\
What You Need to Do:\
•Read Parquet data from Bronze\
•Enforce schema and data types\
•Handle duplicates and null values\
•Create managed Delta tables

In [0]:
%sql
USE CATALOG main;

In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import *

#-------------------------------------------------Reading all files from Bronze layer---------------------------------------------------------
df_customers = spark.read.format("parquet").load("/Volumes/main/ecommerce/lakehouse_vol/bronze/customers/")
df_order = spark.read.parquet("/Volumes/main/ecommerce/lakehouse_vol/bronze/orders/")
df_order_items = spark.read.parquet("/Volumes/main/ecommerce/lakehouse_vol/bronze/order_items/")
df_payments = spark.read.parquet("/Volumes/main/ecommerce/lakehouse_vol/bronze/payments/")
df_products = spark.read.parquet("/Volumes/main/ecommerce/lakehouse_vol/bronze/products/")
# df_customers.distinct().count()
#df_order.distinct().count()
#df_order_items.distinct().count()
# df_payments.distinct().count()
# df_products.distinct().count()
#df_customers.distinct().display()
#df_customers.count()-df_customers.distinct().count()


In [0]:
#-------------------------------------------------------Customers data set-------------------------------------------------------------------
df_customers_clean = (df_customers
    .withColumn("customer_zip_code_prefix",col("customer_zip_code_prefix").cast("int")))

#-------------------------------------------------------Order data set-------------------------------------------------------------------

df_order_clean = (df_order
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("Timestamp"))
    .withColumn("order_approved_at", col("order_approved_at").cast("Timestamp"))
    .withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("Timestamp"))
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("Timestamp"))
    .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("Timestamp")))

#-------------------------------------------------------Items data set-------------------------------------------------------------------

df_items_clean = (df_order_items
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn("shipping_limit_date", col("shipping_limit_date").cast("timestamp"))
    .withColumn("price", col("price").cast(DecimalType(10,2)))
    .withColumn("freight_value", col("freight_value").cast(DecimalType(10,2))))

#-------------------------------------------------------Payment data set-------------------------------------------------------------------

df_payments_clean = (
    df_payments
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast(DecimalType(10,2)))   
)

#-------------------------------------------------------Payment data set-------------------------------------------------------------------

df_products_clean = (
    df_products
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", col("product_weight_g").cast(DecimalType(10,2)))
    .withColumn("product_length_cm", col("product_length_cm").cast(DecimalType(10,2)))
    .withColumn("product_height_cm", col("product_height_cm").cast(DecimalType(10,2)))
    .withColumn("product_width_cm", col("product_width_cm").cast(DecimalType(10,2))) 
)

In [0]:
df_customers_clean.dropDuplicates(['customer_id'])
df_order_clean.dropDuplicates(["order_id"])
df_items_clean.dropDuplicates(["order_id", "order_item_id"])
df_payments_clean.dropDuplicates(["order_id","payment_sequential"])
df_products_clean.dropDuplicates(["product_id"])

#----------------------------------------------------------Droping Null----------------------------------------------------------------------

df_products_cl=df_products_clean.dropna(subset=["product_category_name"])
df_customers_cl=df_customers_clean.dropna(subset=["customer_id"])
df_order_cl=df_order_clean.dropna(subset=["order_id","customer_id"])
df_order_items_cl=df_items_clean.dropna(subset=["order_id"])
df_payments_cl=df_payments_clean.dropna(subset=["order_id"])

In [0]:
df_products_cl.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.sil_products")
df_customers_cl.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.sil_customers")
df_order_cl.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.sil_orders")
df_order_items_cl.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.sil_order_items")
df_payments_cl.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.sil_payments")

In [0]:

#Initial code when there is no data on delta table no use 
from pyspark.sql.functions import *
if spark.catalog.tableExists("main.ecommerce.dim_products_scd") and spark.catalog.tableExists("main.ecommerce.dim_customers_scd") :
    True
else:
    df_scd_customers=df_customers_cl.withColumn("effective_start_date",current_date())\
                             .withColumn("effective_end_date",lit('9999-12-31').cast("date"))\
                             .withColumn("is_Active_flag",lit('Y'))

    df_scd_products=df_products_cl.withColumn("effective_start_date",current_date())\
                       .withColumn("effective_end_date",lit('9999-12-31').cast("date"))\
                       .withColumn("is_Active_flag",lit('Y'))
# display(df_scd_products.limit(5))
# display(df_scd_customers.limit(5))
# df_scd_products.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/dim_products")
# df_scd_customers.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/dim_customers")
#-----------------------------------------------in delta manager------------------------------------------------------
    df_scd_products.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.dim_products_scm")
    df_scd_customers.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.dim_customers_scm")

#SCD-2 Logic

This will be apply on 2 table\
1.customer\
2.Product \
 In that we will add following columns to keep track \
 >effective_start_date\
 >effective_end_date\
 >is_Active flag (true for currentrecords)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#---------------------------------------------------NEW DATA sorce-------------------------------------------------------------------------

df_src_customers=df_customers_cl
df_src_products=df_products_cl

#---------------------------------------------------DIM Table to compere------------------------------------------------------------------

df_tgt_dim_customers=spark.table("main.ecommerce.dim_customers_scm")
df_tgt_dim_products=spark.table("main.ecommerce.dim_products_scm")

#df_customers.printSchema()
#display(df_customers.limit(10))

In [0]:
# Active customers from dimension
#-------------------------------------customers--------------------------------------
tgt_customers=df_tgt_dim_customers.filter(col("is_Active_flag")=="Y")

#-------------------------------------products--------------------------------------
tgt_products=df_tgt_dim_products.filter(col("is_Active_flag")=="Y")

In [0]:
#Help to deteact Existing and New records

#---------------------------------------------------------------------------------------------------------------

join_df_cust=df_src_customers.alias("src").join(tgt_customers.alias("tgt"),on="customer_id",how="left")

#----------------------------------------------------------------------------------------------------------------

join_df_pro=df_src_products.alias("src").join(tgt_products.alias("tgt"),on="product_id",how="left")

# display(join_df_cust)
# display(join_df_pro)


In [0]:
#checking whether src is not same as tgt
#------------------------------------------------------------------------------------------------------------------------------

changed_df_cust=join_df_cust.filter((col("tgt.customer_id").isNotNull()) & 
                                    (col("src.customer_zip_code_prefix") != col("tgt.customer_zip_code_prefix") )|
                                    (col("src.customer_city") != col("tgt.customer_city")) |
                                    (col("src.customer_state") != col("tgt.customer_state")))

#--------------------------------------------------------------------------------------------------------------------------------

changed_df_pro=join_df_pro.filter((col("tgt.product_id").isNotNull()) & 
                                  (col("src.product_photos_qty") != col("tgt.product_photos_qty")) |
                                  (col("src.product_weight_g") != col("tgt.product_weight_g")) | 
                                  (col("src.product_length_cm") != col("tgt.product_length_cm")) | 
                                  (col("src.product_height_cm") != col("tgt.product_height_cm")) | 
                                  (col("src.product_width_cm") != col("tgt.product_width_cm")))


In [0]:
#we are taking tgt records which are changed and making them as expired
#--------------------------------------------------------------------------------------------------------------------------------
expiary_df_cust=changed_df_cust.select("tgt.*").withColumn("effective_end_date",current_date())\
                                               .withColumn("is_Active_flag",lit("N"))

#--------------------------------------------------------------------------------------------------------------------------------

expiary_df_pro=changed_df_pro.select("tgt.*").withColumn("effective_end_date",current_date())\
                                               .withColumn("is_Active_flag",lit("N"))
# display(expiary_df_cust)
# display(expiary_df_pro)


In [0]:
# here we select sec records and which r new and giving them new columns
#------------------------------------------------------------------------------------------------------------------------------
new_version_df_cust=changed_df_cust.select(("src.*"),current_date().alias("effective_start_date"),
                                                     lit("9999-12-31").cast("date").alias("effective_end_date"),
                                                     lit("Y").alias("is_Active_flag"))

#------------------------------------------------------------------------------------------------------------------------------

new_version_df_pro=changed_df_pro.select(("src.*"),current_date().alias("effective_start_date"),
                                                   lit("9999-12-31").cast("date").alias("effective_end_date"),
                                                   lit("Y").alias("is_Active_flag"))

# display(new_version_df_cust)

# display(new_version_df_pro)

In [0]:
#Here we are taking new records where tgt is null but src is not null

#------------------------------------------------------------------------------------------------------------------------------

new_df_cust=join_df_cust.filter(col("tgt.customer_id").isNull()).select(("src.*"),current_date().alias("effective_start_date"),
                                                                   lit("9999-12-31").cast("date").alias("effective_end_date"),
                                                                   lit("Y").alias("is_Active_flag"))

#------------------------------------------------------------------------------------------------------------------------------

new_df_pro=join_df_pro.filter(col("tgt.product_id").isNull()).select(("src.*"),
                                                                   current_date().alias("effective_start_date"),
                                                                   lit("9999-12-31").cast("date").alias("effective_end_date"),
                                                                   lit("Y").alias("is_Active_flag"))

# display(new_df_cust)
# display(new_df_pro)


In [0]:
# antijoin will select left table (tgt) records and which are not changed

#------------------------------------------------------------------------------------------------------------------------------
unchanged_df_cust=df_tgt_dim_customers.join(changed_df_cust.select("customer_id"),on="customer_id",how="leftanti")

#------------------------------------------------------------------------------------------------------------------------------

unchanged_df_pro=df_tgt_dim_products.join(changed_df_pro.select("product_id"),on="product_id",how="leftanti")

# display(unchanged_df_cust.limit(10))
# display(unchanged_df_pro.limit(10))

In [0]:
#------------------------------------------------------------------------------------------------------------------------------

final_dim_cust_df=(unchanged_df_cust.union(expiary_df_cust)
                                    .union(new_version_df_cust)
                                    .union(new_df_cust))
#------------------------------------------------------------------------------------------------------------------------------                                
final_dim_pro_df=(unchanged_df_pro.union(expiary_df_pro)
                                    .union(new_version_df_pro)
                                    .union(new_df_pro))

# display(final_dim_cust_df.limit(10))
# display(final_dim_pro_df.limit(10))
# final_dim_cust_df.write.mode("overwrite").saveAsTable("dbdemos.dim_customer")
# final_dim_pro_df.write.mode("overwrite").saveAsTable("dbdemos.dim_product")

In [0]:
final_dim_pro_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.ecommerce.dim_products")
final_dim_cust_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.ecommerce.dim_customers")


# final_dim_cust_df.write.mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/dim_products")
# final_dim_pro_df.write.mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/dim_customers")

In [0]:
# df_products_cl.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/del_products")
# df_customers_cl.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/del_customers")
# df_order_cl.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/del_orders")
# df_order_items_cl.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/del_order_items")
# df_payments_cl.write.format("delta").mode("overwrite").save("/Volumes/main/ecommerce/lakehouse_vol/silver/del_payments")